# MCP

In [1]:
import os

import chromadb
import dotenv
from agents import Agent, Runner, function_tool, trace, WebSearchTool
from agents.mcp import MCPServerStreamableHttp

dotenv.load_dotenv()

True

Let's set up our RAG database connection:

In [2]:
chroma_client = chromadb.PersistentClient(path="../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [3]:
# This is the same code as in the rag.ipynb notebook


@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Integrate EXA Search as an MCP:

In [4]:
# Exa Search MCP code comes here:
exa_search_mcp = MCPServerStreamableHttp(
    name="Exa Search MCP", 
    params={
        "url":f"https://mcp.exa.ai/mcp?exaApiKey={os.environ.get('EXA_API_KEY')}", 
        "timeout": 30, 
    }, 
    cache_tools_list=True, 
    client_session_timeout_seconds= 30,  
    max_retry_attempts=1
)
await exa_search_mcp.connect()
calorie_agent_with_search = Agent(
    name="Nutrition Assistant",
    instructions="""
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers.
    * You follow this workflow:
        0) First, use the calorie_lookup_tool to get the calorie information of the ingredients. But only use the result if it's explicitly for the food requested in the query.
        1) If you couldn't find the exact match for the food or you need to look up the ingredients, search the EXA web to figure out the exact ingredients of the meal.
        Even if you have the calories in the web search response, you should still use the calorie_lookup_tool to get the calorie
        information of the ingredients to make sure the information you provide is consistent.
        2) Then, if necessary, use the calorie_lookup_tool to get the calorie information of the ingredients.
    * Even if you know the recipe of the meal, always use Exa Search to find the exact recipe and ingredients.
    * Once you know the ingredients, use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * If the query is about the meal, in your final output give a list of ingredients with their quantities and calories for a single serving. Also display the total calories.
    * Don't use the calorie_lookup_tool more than 10 times.
    """,
    tools=[calorie_lookup_tool, WebSearchTool()],
    # mcp_servers=[exa_search_mcp]
)

In [5]:
calorie_agent_with_search_open = Agent(
    name="Nutrition Assistant web search OpenAI",
    instructions="""
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers.
    * You follow this workflow:
        0) First, use the calorie_lookup_tool to get the calorie information of the ingredients. But only use the result if it's explicitly for the food requested in the query.
        1) If you couldn't find the exact match for the food or you need to look up the ingredients, search with webSearchTool to figure out the exact ingredients of the meal.
        Even if you have the calories in the web search response, you should still use the calorie_lookup_tool to get the calorie
        information of the ingredients to make sure the information you provide is consistent.
        2) Then, if necessary, use the calorie_lookup_tool to get the calorie information of the ingredients.
    * Even if you know the recipe of the meal, always use Exa Search to find the exact recipe and ingredients.
    * Once you know the ingredients, use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * If the query is about the meal, in your final output give a list of ingredients with their quantities and calories for a single serving. Also display the total calories.
    * Don't use the calorie_lookup_tool more than 10 times.
    """,
    tools=[calorie_lookup_tool, WebSearchTool()],
    # mcp_servers=[exa_search_mcp]
)

Reference query - shouldn't use ExaSearch:

In [5]:
with trace("Nutrition Assistant with MCP - Only uses calorie_lookup_tool"):
    result = await Runner.run(
        calorie_agent_with_search,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Calories per 100 g:
    - Banana: 89 kcal/100 g
    - Apple: 52 kcal/100 g
    
    To give a total for “a banana and an apple,” please provide the weights (in g) of each fruit or confirm standard sizes (e.g., one medium banana and one medium apple).
- 7 new item(s)
- 2 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [8]:
with trace("Nutrition Assistant with MCP "):
    result = await Runner.run(
        calorie_agent_with_search, "How many calories are in an english breakfast?"
    )
    print(result.final_output)

A Full English breakfast varies by ingredients, but a typical plate is roughly 750–1,200 calories per serving depending on portions and fried items. Components like eggs, bacon, sausages, black pudding, tomatoes, mushrooms, beans, and fried bread can push the total up or down. If you want a precise total, tell me the exact ingredients and amounts (or a specific restaurant recipe) and I’ll calculate it. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Full_breakfast?utm_source=openai))


In [7]:
## Trace for the new agent 

with trace("Nutrition Assistant with Search web Open "):
    result = await Runner.run(
        calorie_agent_with_search_open, "How many calories are in an english breakfast?"
    )
    print(result.final_output)

A traditional Full English breakfast varies by portion and cooking method, but a typical single serving is roughly 900–1,000 calories. 

Example (approximate, common portions):
- 2 eggs (fried/poached): ~140 kcal
- 2 bacon rashers: ~80–90 kcal
- 2 sausages: ~180–240 kcal
- Baked beans (1/2 cup): ~150 kcal
- Toast or fried bread (2 slices): ~150–200 kcal
- Hash browns (1–2): ~120–180 kcal
- Grilled tomato and mushrooms: ~40–60 kcal

Total: around 900–1,000 kcal. Adjust up or down with portion size and cooking fats.
